# PyTorch GPU Colab Example

This notebook demonstrates the Google Colab execution mode used by the **Resources** page.

1. Sign in to Google Colab.
2. Choose **Runtime > Change runtime type > GPU** if a GPU is not already selected.
3. Run the cells from top to bottom, or choose **Runtime > Run all**.

The first code cell requests this Resource's attachment manifest from `jspcv.com` and downloads every listed attachment into the temporary `attachments` directory in your private Colab runtime. The files disappear when the Colab runtime is deleted.

In [ ]:
import json
from pathlib import Path
from urllib.parse import quote
from urllib.request import Request, urlopen

SITE_URL = "https://jspcv.com"
NOTEBOOK_FILENAME = "pytorch-gpu-colab-example.ipynb"
MANIFEST_URL = (
    f"{SITE_URL}/resources/colabAttachments/"
    f"{quote(NOTEBOOK_FILENAME, safe='')}/manifest.json"
)
ATTACHMENT_DIR = Path("attachments")
ATTACHMENT_DIR.mkdir(exist_ok=True)

request = Request(MANIFEST_URL, headers={"User-Agent": "jspcv-colab-example"})
with urlopen(request, timeout=30) as response:
    manifest = json.load(response)

for attachment in manifest["attachments"]:
    target = ATTACHMENT_DIR / Path(attachment["name"]).name
    request = Request(attachment["url"], headers={"User-Agent": "jspcv-colab-example"})
    with urlopen(request, timeout=60) as response:
        target.write_bytes(response.read())
    print(f"Downloaded: {target}")

print(f"Ready: {len(manifest['attachments'])} attachment(s)")

## Verify PyTorch and the GPU

This cell stops with a clear message when the notebook is connected to a CPU runtime.

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is assigned. Choose Runtime > Change runtime type > GPU, "
        "reconnect, and run the notebook again."
    )

device = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA runtime: {torch.version.cuda}")

## Train a small neural network

The attached JSON file controls the random seed, dataset size, network size, epoch count, and learning rate. Synthetic data keeps this example small while still exercising PyTorch tensor operations on the assigned GPU.

In [ ]:
import json
from torch import nn

config = json.loads((ATTACHMENT_DIR / "colab-gpu-example-config.json").read_text())
torch.manual_seed(config["seed"])

generator = torch.Generator().manual_seed(config["seed"])
x_cpu = torch.randn(config["samples"], config["features"], generator=generator)
teacher = torch.randn(config["features"], config["classes"], generator=generator)
y_cpu = (x_cpu @ teacher).argmax(dim=1)
x = x_cpu.to(device)
y = y_cpu.to(device)

model = nn.Sequential(
    nn.Linear(config["features"], config["hidden_units"]),
    nn.ReLU(),
    nn.Linear(config["hidden_units"], config["classes"]),
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
criterion = nn.CrossEntropyLoss()

print(config)
print(f"Training tensors: {x.device} / {y.device}")

In [ ]:
import time

loss_history = []
torch.cuda.synchronize()
started_at = time.perf_counter()

for epoch in range(1, config["epochs"] + 1):
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:02d} | loss={loss.item():.4f}")

torch.cuda.synchronize()
elapsed = time.perf_counter() - started_at
with torch.inference_mode():
    accuracy = (model(x).argmax(dim=1) == y).float().mean().item()

print(f"Training time: {elapsed:.3f} seconds")
print(f"Training accuracy: {accuracy:.2%}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3.5))
plt.plot(range(1, len(loss_history) + 1), loss_history, color="#0969da")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title(f"PyTorch training on {torch.cuda.get_device_name(0)}")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()